In [3]:
import pandas as pd
import numpy as np

# 1. Llegir els arxius (assegura't que els CSV estan a la mateixa carpeta)
estacions_df = pd.read_csv('estacions.csv')
trams_df = pd.read_csv('trams_linia_metro.csv')

# 2. Crear el diccionari d'estacions (Nom -> ID)
# Netejem els espais en blanc per evitar problemes ("  Clot " -> "Clot")
noms_estacions = estacions_df['NOM_ESTACIO'].str.strip().unique()

estacio_a_id = {nom: i for i, nom in enumerate(noms_estacions)}
id_a_estacio = {i: nom for nom, i in estacio_a_id.items()}
N = len(noms_estacions) # Nombre total d'estacions úniques (140)

print(f"S'han detectat {N} estacions úniques.")

# 3. Inicialitzar la matriu d'adjacència amb zeros
A = np.zeros((N, N), dtype=int)

# 4. Emplenar la matriu iterant per les arestes (trams)
for index, row in trams_df.iterrows():
    origen = str(row['NOM_ESTACIO_INI']).strip()
    desti = str(row['NOM_ESTACIO_FI']).strip()
    
    # Descartem els nodes fantasma "Inici" i "Final"
    if origen != "Inici" and desti != "Final":
        # Comprovem que ambdues estacions existeixen al nostre diccionari
        if origen in estacio_a_id and desti in estacio_a_id:
            i = estacio_a_id[origen]
            j = estacio_a_id[desti]
            
            # Posem un 1 en ambdues direccions (graf no dirigit)
            A[i, j] = 1
            A[j, i] = 1

print("Matriu d'adjacència construïda amb èxit!")
print("-" * 50)

# 5. Càlcul del recorregut més curt a partir de les potències de la matriu
def troba_recorregut_mes_curt(origen_nom, desti_nom, matriu_A, diccionari):
    if origen_nom not in diccionari or desti_nom not in diccionari:
        return "Error: Alguna de les estacions no existeix a la base de dades."
    
    i = diccionari[origen_nom]
    j = diccionari[desti_nom]
    
    # Si demanem la distància a la mateixa estació
    if i == j:
        return 0, 1
        
    M = matriu_A.copy()
    k = 1  # Llargada del recorregut
    N_nodes = matriu_A.shape[0]
    
    # Anem multiplicant fins trobar un valor > 0 a la casella (i,j) 
    # Ens assegurem de no superar el nombre màxim de nodes possibles (N_nodes)
    while M[i, j] == 0 and k < N_nodes:
        k += 1
        M = M @ matriu_A  # Equival a fer A^k multiplicant pas a pas
        
    if M[i, j] > 0:
        nombre_camins = M[i, j]
        return k, nombre_camins
    else:
        return -1, 0  # Si recorrem tot el graf i segueix a 0, no estan connectades



S'han detectat 140 estacions úniques.
Matriu d'adjacència construïda amb èxit!
--------------------------------------------------


In [6]:
# === ZONA DE PROVES ===

origen_test = "Mas Blau"
desti_test = "Ciutat Meridiana"

resultat = troba_recorregut_mes_curt(origen_test, desti_test, A, estacio_a_id)

if isinstance(resultat, str):
    print(resultat) # Imprimeix l'error d'estacions inexistents
else:
    k_min, num_recorreguts = resultat
    
    print(f"Estació Origen: {origen_test} (ID: {estacio_a_id.get(origen_test)})")
    print(f"Estació Destí:  {desti_test} (ID: {estacio_a_id.get(desti_test)})")
    
    if k_min == -1:
        print(f"No s'ha trobat cap camí entre aquestes estacions.")
    else:
        print(f"-> El recorregut MÉS CURT té una llargada de: {k_min} trams (parades).")
        print(f"-> Hi ha {num_recorreguts} camí/ns diferent/s que permeten fer aquest trajecte més curt.")

Estació Origen: Mas Blau (ID: 107)
Estació Destí:  Ciutat Meridiana (ID: 136)
-> El recorregut MÉS CURT té una llargada de: 28 trams (parades).
-> Hi ha 2 camí/ns diferent/s que permeten fer aquest trajecte més curt.
